# 01 — SQL Pipeline Analysis

This notebook calculates pipeline KPIs from `data/sales_pipeline.csv` using pandas and SQLite. It includes SQL CTEs and window functions for agent ranking.

**Expected fields:** `opportunity_id`, `sales_agent`, `product`, `stage`, `revenue`; `close_date` is optional. See `README.md` for definitions.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

DATA_PATH = Path("data/sales_pipeline.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Add your source CSV at data/sales_pipeline.csv. "
        "Do not use fabricated data for the project results."
    )

df = pd.read_csv(DATA_PATH)
# If your source has different headers, rename them here, for example:
# df = df.rename(columns={"Deal Stage": "stage", "Close Value": "revenue"})

required = {"opportunity_id", "sales_agent", "product", "stage", "revenue"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}. Rename source headers in the cell above.")

df["stage"] = df["stage"].astype("string").str.strip()
df["revenue"] = pd.to_numeric(df["revenue"], errors="coerce").fillna(0)
df["stage_norm"] = df["stage"].str.lower()
df["is_won"] = df["stage_norm"].eq("won")
df["is_lost"] = df["stage_norm"].eq("lost")
df["is_closed"] = df["is_won"] | df["is_lost"]
df["is_active"] = ~df["is_closed"]

print(f"Rows: {len(df):,}")
print(f"Unique opportunity IDs: {df['opportunity_id'].nunique():,}")
print("Stage counts:")
display(df["stage"].value_counts(dropna=False).rename_axis("stage").to_frame("opportunities"))

## Load the cleaned data into SQLite

In [ ]:
con = sqlite3.connect(":memory:")
df.to_sql("opportunities", con, if_exists="replace", index=False)

overall_sql = '''
WITH pipeline AS (
    SELECT
        COUNT(*) AS total_opportunities,
        SUM(CASE WHEN LOWER(stage) = 'won' THEN 1 ELSE 0 END) AS won_opportunities,
        SUM(CASE WHEN LOWER(stage) = 'lost' THEN 1 ELSE 0 END) AS lost_opportunities,
        SUM(CASE WHEN LOWER(stage) NOT IN ('won', 'lost') THEN 1 ELSE 0 END) AS active_deals,
        SUM(CASE WHEN LOWER(stage) = 'won' THEN revenue ELSE 0 END) AS won_revenue
    FROM opportunities
),
metrics AS (
    SELECT *,
        won_opportunities + lost_opportunities AS closed_opportunities
    FROM pipeline
)
SELECT *,
    ROUND(100.0 * won_opportunities / NULLIF(closed_opportunities, 0), 2) AS closed_win_rate_pct
FROM metrics;
'''
overall = pd.read_sql_query(overall_sql, con)
display(overall)

## Rank sales agents with a SQL window function

The ranking below is based on closed-opportunity win rate, with closed volume as a tie-breaker. Agents without closed deals remain at the bottom.

In [ ]:
agent_sql = '''
WITH agent_stats AS (
    SELECT
        sales_agent,
        SUM(CASE WHEN LOWER(stage) = 'won' THEN 1 ELSE 0 END) AS won_opportunities,
        SUM(CASE WHEN LOWER(stage) = 'lost' THEN 1 ELSE 0 END) AS lost_opportunities,
        SUM(CASE WHEN LOWER(stage) IN ('won', 'lost') THEN 1 ELSE 0 END) AS closed_opportunities,
        ROUND(
            100.0 * SUM(CASE WHEN LOWER(stage) = 'won' THEN 1 ELSE 0 END)
            / NULLIF(SUM(CASE WHEN LOWER(stage) IN ('won', 'lost') THEN 1 ELSE 0 END), 0),
            2
        ) AS win_rate_pct,
        SUM(CASE WHEN LOWER(stage) = 'won' THEN revenue ELSE 0 END) AS won_revenue
    FROM opportunities
    GROUP BY sales_agent
),
ranked AS (
    SELECT *,
        RANK() OVER (
            ORDER BY win_rate_pct DESC, closed_opportunities DESC
        ) AS performance_rank
    FROM agent_stats
)
SELECT *
FROM ranked
ORDER BY performance_rank, sales_agent;
'''
agent_performance = pd.read_sql_query(agent_sql, con)
display(agent_performance.head(30))

## Product-level pipeline performance

In [ ]:
product_sql = '''
SELECT
    product,
    COUNT(*) AS total_opportunities,
    SUM(CASE WHEN LOWER(stage) NOT IN ('won', 'lost') THEN 1 ELSE 0 END) AS active_deals,
    SUM(CASE WHEN LOWER(stage) = 'won' THEN 1 ELSE 0 END) AS won_opportunities,
    SUM(CASE WHEN LOWER(stage) IN ('won', 'lost') THEN 1 ELSE 0 END) AS closed_opportunities,
    ROUND(
        100.0 * SUM(CASE WHEN LOWER(stage) = 'won' THEN 1 ELSE 0 END)
        / NULLIF(SUM(CASE WHEN LOWER(stage) IN ('won', 'lost') THEN 1 ELSE 0 END), 0),
        2
    ) AS win_rate_pct,
    SUM(CASE WHEN LOWER(stage) = 'won' THEN revenue ELSE 0 END) AS won_revenue
FROM opportunities
GROUP BY product
ORDER BY won_revenue DESC;
'''
product_performance = pd.read_sql_query(product_sql, con)
display(product_performance)

## Check key resume metrics against the current source file

This cell prints the computed values for comparison. Differences can come from a different data version, filters, or metric definitions; investigate those before publishing results.

In [ ]:
kpi = overall.iloc[0]
print(f"Total opportunities: {int(kpi['total_opportunities']):,}")
print(f"Closed win rate: {kpi['closed_win_rate_pct']}%")
print(f"Won revenue: ${kpi['won_revenue']:,.2f}")
print(f"Active deals: {int(kpi['active_deals']):,}")

for product_name in ["GTX Pro", "MG Special"]:
    match = product_performance[product_performance["product"].astype(str).str.casefold() == product_name.casefold()]
    if match.empty:
        print(f"{product_name}: not found in product column")
    else:
        row = match.iloc[0]
        print(f"{product_name}: won revenue=${row['won_revenue']:,.2f}; win rate={row['win_rate_pct']}%; active deals={int(row['active_deals']):,}")

## Export analysis tables

These files can be used directly in the Tableau-preparation notebook or connected to Tableau.

In [ ]:
EXPORT_DIR = Path("data/tableau_exports")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

overall.to_csv(EXPORT_DIR / "overall_kpis.csv", index=False)
agent_performance.to_csv(EXPORT_DIR / "agent_performance.csv", index=False)
product_performance.to_csv(EXPORT_DIR / "product_performance.csv", index=False)

print(f"Saved KPI, agent, and product tables to: {EXPORT_DIR.resolve()}")
con.close()